In [ ]:
# 📄 CONFIGURATION - Remplacez par l'URL de votre Google Sheet
SHEET_URL = "https://docs.google.com/spreadsheets/...."

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_NAME = "cscp-positionning-solver"
REPO_URL  = "https://github.com/sylcordo/cscp-positionning-solver.git"

def run_cmd(cmd, cwd=None, check=True):
    print(f"> {' '.join(cmd)} (cwd={cwd or Path.cwd()})")
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout.strip())
    if proc.stderr:
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
    return proc

def remove_path(target: Path):
    if not target.exists():
        print(f"→ {target} n'existe pas, rien à supprimer.")
        return
    # Sécurité : ne supprimer que si le nom du dossier correspond exactement à REPO_NAME
    if target.name != REPO_NAME:
        raise RuntimeError(f"Refus de supprimer {target} : le nom du dossier doit être exactement '{REPO_NAME}'.")
    print(f"→ Suppression récursive de {target} ...")
    # si on est dans le dossier lui-même, on devra d'abord remonter
    cwd = Path.cwd().resolve()
    target_resolved = target.resolve()
    if cwd == target_resolved:
        os.chdir(target_resolved.parent)
        print(f"→ Nous étions dans le dossier du repo ; changement de répertoire vers {Path.cwd()}")
    # suppression
    try:
        shutil.rmtree(target_resolved)
        print("→ Suppression terminée.")
    except Exception as e:
        print("Erreur lors de la suppression :", e)
        raise

def main():
    cwd = Path.cwd()
    repo_path = cwd / REPO_NAME

    # Cas 1: on est déjà dans le repo (cwd.name == REPO_NAME) -> remonter d'un niveau puis supprimer
    if cwd.name == REPO_NAME:
        print(f"→ Le répertoire courant est {REPO_NAME}. Nous allons le supprimer et recloner.")
        remove_path(cwd)

    # Cas 2: le dossier existe au même niveau -> supprimer
    elif repo_path.exists():
        print(f"→ {repo_path} existe. Nous allons le supprimer et recloner.")
        remove_path(repo_path)

    # Cas 3: le dossier n'existe pas -> on clone directement
    else:
        print("→ Aucun dossier local trouvé. Pas de suppression nécessaire, on clone directement.")

    # Clone proprement
    target_parent = Path.cwd()
    clone_path = target_parent / REPO_NAME
    try:
        run_cmd(["git", "clone", REPO_URL, str(clone_path)])
    except subprocess.CalledProcessError as e:
        print("Échec du clone :")
        print(e.stderr or e.output)
        sys.exit(1)

    # Se placer dans le repo cloné
    os.chdir(clone_path)
    print(f"\n📂 Répertoire courant : {Path.cwd()}\n")

    # Installation dépendances (optionnel - commente si inutile)
    print("📦 Installation des dépendances (requirements.txt)...")
    try:
        run_cmd([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"])
        print("✅ Installation terminée!\n")
    except subprocess.CalledProcessError as e:
        print("⚠️ Installation des dépendances échouée :")
        print(e.stderr or e.output)

    # 🎯 Lancement
    print("🎯 Lancement de l'optimisation...\n")
    from run_colab import run_optimization

    run_optimization(sheet_url=SHEET_URL)

if __name__ == "__main__":
    main()
